# MolForge GRPO Training Pipeline
This notebook implements the Reinforcement Learning (GRPO) training pipeline for the MolForge environment.
We train the model using a **Proposer-Critic-Selector** architecture and targeted **reward shaping** to overcome local minima.

In [ ]:
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U "trl>=0.21.0" peft accelerate bitsandbytes datasets matplotlib pandas huggingface_hub "openenv-core[core]>=0.2.3" rdkit jmespath xformers

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = os.getenv("MOLFORGE_REPO_URL", "https://github.com/Adhitya-Vardhan/molt_lab.git")
REPO_BRANCH = os.getenv("MOLFORGE_GIT_BRANCH", "codex/work-2026-05-01")
LOCAL_REPO = os.getenv("MOLFORGE_LOCAL_REPO", "")
REPO_DIR = Path(os.getenv("MOLFORGE_REPO_DIR", "/content/molt_lab"))

def run_git(*args: str) -> None:
    subprocess.run(["git", *args], check=True)

if LOCAL_REPO and Path(LOCAL_REPO).exists():
    REPO_DIR = Path(LOCAL_REPO).resolve()
else:
    if not REPO_DIR.exists():
        run_git("clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR))
    else:
        run_git("-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH)
        run_git("-C", str(REPO_DIR), "checkout", REPO_BRANCH)
        run_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(str(REPO_DIR))
print(f"Using MolForge repo: {REPO_DIR}")
print(f"Requested branch: {REPO_BRANCH}")

In [ ]:
import time
import os

# Training Configuration
os.environ["MOLFORGE_REWARD_MODE"] = "curriculum"
os.environ["MOLFORGE_TRAINING_RANDOMIZATION"] = "1"

RL_MAX_STEPS = 50
NUM_GENERATIONS = 2
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-6
MAX_SEQ_LENGTH = 2048
MAX_PROMPT_LENGTH = 1536
MAX_COMPLETION_LENGTH = 384

RUN_NAME = time.strftime("molforge_grpo_%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"/content/molforge_rl_runs/{RUN_NAME}")
ADAPTER_SAVE_DIR = OUTPUT_DIR / "adapters"
PLOT_DIR = OUTPUT_DIR / "plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


### Reward Function & OpenEnv Integration
We implement a custom reward function that wraps the native `MolForgeEnvironment`. 
To prevent "reward hacking" (where the model endlessly farms `run_assay` for safe points), we apply targeted reward shaping.

In [ ]:
from hf_submission.rl_training_helpers import molforge_reward_func


### Model & Tokenizer Loading

In [ ]:
from unsloth import FastLanguageModel

# Set this to your SFT checkpoint Deployed to hugging face space 
# SFT trained on only to mimic the response behavioiur of the model (structured responses visit the hf blog for more detailed explanation )
SFT_ADAPTER_PATH = "Adhitya122/qwen3_5_2b_molforge_sft_v4"

print("Loading model and applying Unsloth optimizations...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Enable fast training paths
FastLanguageModel.for_training(model)

# Extract underlying tokenizer if it is wrapped in a vision processor
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

### GRPO Training Loop

In [ ]:
from hf_submission.rl_training_helpers import build_dynamic_prompts

dataset = build_dynamic_prompts(episodes=20, max_turns=6, randomized=True, seed="dynamic-rl")
print(f"Dynamic dataset created with {len(dataset)} prompt states.")


In [ ]:
from trl import GRPOConfig, GRPOTrainer
import inspect
import torch

# Check for BF16 support (T4 does not support it, A100/L4 do)
has_bf16 = torch.cuda.is_bf16_supported()
print(f"GPU supports BF16: {has_bf16}")

config_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_completion_length": MAX_COMPLETION_LENGTH,
    "num_generations": NUM_GENERATIONS,
    "max_steps": RL_MAX_STEPS,
    "logging_steps": 1,
    "save_steps": 25,
    "bf16": has_bf16,
    "fp16": not has_bf16,
    "report_to": "none",
    "log_completions": True,
}

supported_params = inspect.signature(GRPOConfig.__init__).parameters
filtered_kwargs = {k: v for k, v in config_kwargs.items() if k in supported_params}

training_args = GRPOConfig(**filtered_kwargs)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=molforge_reward_func,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO Training...")
train_result = trainer.train()

print(f"Training complete. Saving adapters to {ADAPTER_SAVE_DIR}")
trainer.save_model(str(ADAPTER_SAVE_DIR))
tokenizer.save_pretrained(str(ADAPTER_SAVE_DIR))


### Post-Training Analysis

In [ ]:
from hf_submission.rl_training_helpers import COMPLETION_LOG, generate_training_artifacts

plot_paths = generate_training_artifacts(
    log_history=trainer.state.log_history,
    completion_log_path=COMPLETION_LOG,
    output_dir=PLOT_DIR,
)

print("Generated plot artifacts:")
for name, path in plot_paths.items():
    print(f"- {name}: {path}")

plot_paths


In [ ]:
from IPython.display import Image, display

for key in [
    "reward_curve",
    "loss_curve",
    "completion_reward_curve",
    "final_score_curve",
    "action_distribution",
]:
    if key in plot_paths:
        print(key)
        display(Image(filename=plot_paths[key]))
